# 네이버 뉴스 본문 수집 (Selenium 로컬용)

기사 주소 수집 노트북에서 만든 `링크_{press}_*.json`을 읽고, 크롬 창으로 기사를 하나씩 열어 제목·본문·날짜·카테고리를 가져오는 노트북임.
Colab용 `언론사_네이버뉴스_본문_수집_bs4_colab.ipynb`의 로컬 버전이고, 저장 형식(열 구성)은 그대로라 뒤 단계에 그대로 이어짐.
언론사별 기간 파일 하나를 읽어 본문 CSV 하나로 저장함. 중간에 끊겨도 저장된 지점부터 이어서 가능.

- 입력: `링크_{press}_{YYMMDD}_{YYMMDD}.json`
- 출력: `본문_selenium_{press}_{YYMMDD}_{YYMMDD}.csv` (파일 앞말은 아래 `BODY_PREFIX`에서 변경 가능)
- 보조 파일: 중간 저장 JSON, 다시 시도해도 실패한 기사 주소 JSON
- 처리 내용: 기사 본문 수집, 오류 기사 1회 재시도, 스포츠/연예 기사 따로 처리, 이미 끝난 파일 건너뛰기
- Colab판과 다른 점: requests 대신 크롬 창(Selenium)으로 기사 페이지를 염. 화면으로 확인되는 대신 훨씬 느림 (기사당 2~3초)


In [ ]:
# # 로컬 커널에 필요한 도구 설치
# # 이미 깔려 있으면 이 셀은 건너뛰어도 됨
# %pip install -q selenium beautifulsoup4 pandas

In [ ]:
import json
import os
import platform
import random
import re
import shutil
import subprocess
import time
import unicodedata
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

# 기사 페이지를 읽을 때 쓰는 기본 정보 설정 — 주소 수집 노트북과 같은 값 사용
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 크롬 창을 직접 보면서 확인하려면 False, 창 없이 돌리려면 True로 변경
# 로컬에서 눈으로 보려고 만든 노트북이라 기본값은 False
HEADLESS = False

# 저장할 본문 CSV 파일 앞말
# Colab판(BS4)이 만든 '본문_bs4_*'와 구분되게 기본값은 '본문_selenium'
# Colab에서 만든 파일을 그대로 대체하려면 '본문_bs4'로 변경 (아래 통합 셀은 두 이름 다 읽음)
BODY_PREFIX = '본문_selenium'

# 수집할 언론사와 기간 지정
# 기사 주소 수집 노트북과 동일한 press_ranges 그대로 사용. oid는 본문 수집에선 안 씀
# press 하나당 start_date ~ end_date 통합 1개 CSV로 저장
# 날짜 형식: 'YYYY.MM.DD'
press_ranges = [
    # 지상파
    {'press': 'SBS', 'oid': '055', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'KBS', 'oid': '056', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'MBC', 'oid': '214', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    # 경제
    {'press': '한국경제', 'oid': '015', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '매일경제', 'oid': '009', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    # 정치색
    {'press': '한겨레', 'oid': '028', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '조선일보', 'oid': '023', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    # 통신·보도
    {'press': '연합뉴스', 'oid': '001', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'YTN',     'oid': '052', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]


# press_ranges의 항목을 실행할 작업 목록으로 바꿈
# 기사 주소 수집 때와 같은 언론사명, 시작일, 종료일을 본문 수집에도 사용
def build_period_jobs(press_ranges):
    jobs = []
    for item in press_ranges:
        # 'YYYY.MM.DD' 형식의 날짜를 실제 날짜로 바꿔 기간이 맞는지 확인
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작 일자가 끝 일자보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if start > end:
            raise ValueError(f"시작 일자가 끝 일자보다 늦습니다: {item}")
        # oid는 본문 수집에선 안 쓰니까 제외 (링크 파일명은 press로만 매칭)
        jobs.append({
            'press': item['press'],
            'start_date': item['start_date'],
            'end_date': item['end_date'],
        })
    return jobs


# 만든 작업 목록은 아래 수집 셀에서 순서대로 실행
jobs = build_period_jobs(press_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)


# 로컬 프로젝트 폴더 찾기 — 노트북을 어느 폴더에서 열어도 프로젝트 최상위를 가리키게 함
# 현재 폴더부터 위로 올라가며 pipeline_py와 notebooks가 같이 있는 곳을 프로젝트 폴더로 봄
DEFAULT_PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')


def find_project_dir(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pipeline_py').exists() and (candidate / 'notebooks').exists():
            return candidate
    # 못 찾으면 이 컴퓨터의 기본 경로 사용
    return DEFAULT_PROJECT_DIR


PROJECT_DIR = find_project_dir()

# 저장할 폴더 지정 — 링크 파일, 중간 저장 파일, 본문 CSV, 실패 목록이 모두 이 폴더에 저장
# 주소 수집(Selenium 로컬) 노트북과 같은 폴더라 링크 파일을 바로 읽어 옴
SAVE_DIR = PROJECT_DIR / 'data' / 'news' / 'crawling'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f'프로젝트 폴더: {PROJECT_DIR}')
print(f'저장 폴더: {SAVE_DIR}')
print(f'User-Agent: {USER_AGENT}')
print(f'HEADLESS: {HEADLESS}')

In [ ]:
# 현재 컴퓨터에 설치된 크롬 위치 찾기
# 크롬 위치를 직접 알려 주면 실행 오류가 줄어듦
def find_chrome_binary():
    candidates = []

    if platform.system() == 'Windows':
        # 윈도우에서 크롬이 자주 설치되는 폴더들을 후보로 둠
        candidates.extend([
            os.path.expandvars(r'%ProgramFiles%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%LocalAppData%\Google\Chrome\Application\chrome.exe'),
        ])
    else:
        # 리눅스나 WSL에서는 크롬 이름 후보를 차례로 확인
        for name in ['google-chrome', 'google-chrome-stable', 'chromium-browser', 'chromium']:
            found = shutil.which(name)
            if found:
                candidates.append(found)

    # 실제로 존재하는 첫 번째 경로 사용
    for path in candidates:
        if path and Path(path).exists():
            return str(Path(path))
    return None


# 기사 페이지를 열 크롬 준비
# 기본값은 사람이 화면을 보면서 확인할 수 있도록 창을 띄우는 방식임
def build_driver(headless=HEADLESS):
    options = Options()
    options.add_argument(f'user-agent={USER_AGENT}')  # 수집할 때 쓰는 기본 정보 맞춤
    options.add_argument('--lang=ko-KR')  # 가능하면 한국어 페이지로 받기
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 크롬이 자동 실행 창처럼 보이는 표시를 줄임
    options.add_experimental_option('useAutomationExtension', False)  # 크롬이 자동 실행 창처럼 보이는 표시를 줄임
    options.add_argument('--disable-blink-features=AutomationControlled')  # 크롬이 자동 실행 창처럼 보이는 표시를 줄임
    options.add_argument('--window-size=1400,1000')  # 항상 비슷한 화면 크기로 열기

    if headless:
        options.add_argument('--headless=new')  # 창 없이 실행

    if platform.system() != 'Windows':
        options.add_argument('--no-sandbox')  # 리눅스/WSL에서 크롬 실행 오류 줄이기
        options.add_argument('--disable-dev-shm-usage')  # 크롬이 중간에 꺼지는 문제 줄이기
        options.add_argument('--disable-gpu')  # 창 없이 실행할 때 그래픽 관련 오류 줄이기

    # 크롬 실행 파일을 찾으면 직접 지정, 못 찾으면 기본 방식으로 실행
    chrome_binary = find_chrome_binary()
    if chrome_binary:
        options.binary_location = chrome_binary
        print(f'Chrome binary: {chrome_binary}')
        subprocess.run([chrome_binary, '--version'], check=False)
    else:
        print('크롬 실행 파일을 직접 찾지 못해 Selenium 기본 방식으로 실행')

    # 현재 크롬에 맞는 실행 도구로 크롬 창 열기
    driver = webdriver.Chrome(service=Service(), options=options)

    # 네이버가 자동 실행 창이라고 판단할 가능성을 조금 낮춤
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    return driver


driver = build_driver()

In [ ]:
# 네이버에 너무 자주 접속하지 않도록 기사/언론사 사이에 짧게 랜덤 대기
# 크롬 창으로 여는 방식은 그 자체로 느려서 기사 간 대기는 짧게 잡음
ARTICLE_PAUSE_RANGE_SEC = (0.4, 1.2)
JOB_PAUSE_RANGE_SEC = (8, 20)

# 중간에 끊겨도 이어서 수집할 수 있도록 일정 건수마다 중간 저장
# 크롬 방식은 한 건에 시간이 더 걸려서 Colab판(200건)보다 자주 저장
CHECKPOINT_INTERVAL = 50
SKIP_COMPLETED = True

# 기사 페이지를 연 뒤 본문이 뜰 시간 조금 주기
PAGE_LOAD_WAIT_SEC = 0.8
# 제목이나 본문이 바로 안 보일 때 최대 몇 초까지 기다릴지 설정
SELENIUM_WAIT_SEC = 8

# 일반 뉴스 주소가 스포츠/연예 페이지로 자동 이동하는 경우를 따로 처리하기 위한 설정
# 주소에 아래 글자가 들어 있으면 스포츠/연예 기사로 보고 카테고리를 따로 가져옴
# 다른 주소 유형이 생기면 이 표에만 추가하면 됨
REDIRECT_DOMAINS = {
    'sports.naver.com': ('스포츠', r'https://m\.sports\.naver\.com/([^/]+)/article/'),
    'entertain.naver.com': ('연예', r'https://m\.entertain\.naver\.com/([^/]+)/article/'),
}


# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)


# 파일명에 사용할 YYMMDD_YYMMDD 형식 기간 문자열 생성
# 예: 2026.05.01 ~ 2026.05.07 -> '260501_260507'
def make_period_suffix(start_date, end_date):
    return f"{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}"


# 크롬이 넘겨주는 본문은 문단마다 줄바꿈이 들어 있어 한 줄로 정리
# BS4판 결과와 같은 모양으로 맞추려고 줄바꿈은 지우고 앞뒤 공백만 정리
def normalize_body_text(text):
    return text.replace('\n', '').strip()


# 화면에 보이는 한글 날짜를 일반 뉴스 날짜값과 같은 형태로 바꿈
# 예: '2026.05.05. 오전 7:10' -> '2026-05-05 07:10:00'
def parse_korean_datetime(text):
    m = re.match(r'(\d{4})\.(\d{2})\.(\d{2})\.\s*(오전|오후)\s*(\d{1,2}):(\d{2})', text)
    if not m:
        return ''
    y, mo, d, ampm, h, mi = m.groups()
    h = int(h)
    # 12시간제 → 24시간제 변환 (오전 12시 = 00시, 오후 12시 = 12시 그대로)
    if ampm == '오후' and h != 12:
        h += 12
    elif ampm == '오전' and h == 12:
        h = 0
    return f'{y}-{mo}-{d} {h:02d}:{mi}:00'


# 스포츠/연예 기사는 일반 뉴스와 페이지 구조가 달라 따로 추출
# 스포츠/연예 쪽은 제목, 본문, 날짜가 들어 있는 위치가 서로 비슷함
def extract_redirected_article_selenium(driver, original_link):
    current_url = driver.current_url

    # 제목 추출하기 — 페이지 정보에 들어 있는 제목 사용
    title_elements = driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:title"]')
    title = title_elements[0].get_attribute('content').strip() if title_elements else ''

    # 본문 추출하기 — 스포츠/연예 기사 본문 영역 사용
    body_elements = driver.find_elements(By.CSS_SELECTOR, 'div._article_content')
    body = normalize_body_text(body_elements[0].text) if body_elements else ''

    # 날짜 추출하기 — em.date 첫 번째(입력일). 두 번째는 수정일이라 무시
    date_elements = driver.find_elements(By.CSS_SELECTOR, 'em.date')
    pubdate = parse_korean_datetime(date_elements[0].text.strip()) if date_elements else ''

    # 주소에 sports나 entertain이 들어 있는지 보고 카테고리 정리
    # 예: m.sports.naver.com/golf/article/... -> '스포츠/golf'
    category = '기타'
    for domain, (prefix, pattern) in REDIRECT_DOMAINS.items():
        if domain in current_url:
            cat_match = re.match(pattern, current_url)
            category = f'{prefix}/{cat_match.group(1)}' if cat_match else prefix
            break

    # 제목/본문/날짜 중 하나라도 없으면 실패로 기록하고 재시도 대상에 포함
    if not title or not body or not pubdate:
        raise ValueError(f'스포츠/연예 기사: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    # 저장할 때는 처음에 모아 둔 기사 주소 그대로 남김
    return {
        'link': original_link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


# 기사 한 건에서 title/body/pubdate/category 추출 (Selenium 버전)
# 처음에 바로 안 보이면 한 번 더 기다렸다가 다시 확인
def extract_article_selenium(driver, link):
    # 실제 네이버 뉴스 페이지 열기 — 스포츠/연예로 자동 이동되는 경우도 있음
    driver.get(link)

    # 페이지가 열릴 시간 조금 주기
    wait = WebDriverWait(driver, SELENIUM_WAIT_SEC)
    time.sleep(PAGE_LOAD_WAIT_SEC)

    # 스포츠/연예 페이지로 이동된 기사면 위에서 만든 별도 추출 함수 사용
    if any(domain in driver.current_url for domain in REDIRECT_DOMAINS):
        return extract_redirected_article_selenium(driver, link)

    # 제목 추출하기
    title_elements = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
    title = title_elements[0].text.strip() if title_elements else ''

    # 본문 추출하기 — 줄바꿈 정리해 단일 문자열로
    body_elements = driver.find_elements(By.ID, 'newsct_article')
    body = normalize_body_text(body_elements[0].text) if body_elements else ''

    # 날짜 추출하기 — 화면 글자 대신 페이지 안의 data-date-time 날짜값 사용
    pubdate_elements = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
    pubdate = pubdate_elements[0].get_attribute('data-date-time') if pubdate_elements else ''

    # 카테고리 추출하기 — 상단 탭 중 현재 활성화된(aria-selected="true") 항목
    category_elements = driver.find_elements(By.CSS_SELECTOR, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')
    category = category_elements[0].text.strip() if category_elements else ''

    if not title or not body or not pubdate:
        # 본문이 늦게 뜨는 경우가 있어 짧게 기다린 뒤 한 번 더 확인
        wait.until(lambda d: d.find_elements(By.ID, 'newsct_article') or d.find_elements(By.CLASS_NAME, 'media_end_head_headline'))
        title_elements = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
        body_elements = driver.find_elements(By.ID, 'newsct_article')
        pubdate_elements = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')

        # 새로 찾은 값이 있으면 그 값으로 변경
        title = title_elements[0].text.strip() if title_elements else title
        body = normalize_body_text(body_elements[0].text) if body_elements else body
        pubdate = pubdate_elements[0].get_attribute('data-date-time') if pubdate_elements else pubdate

    # 제목/본문/날짜 중 하나라도 없으면 어느 값이 비었는지 표시하고 재시도 대상에 포함
    if not title or not body or not pubdate:
        raise ValueError(f'title/body/pubdate 추출 실패: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    return {
        'link': link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


# 한글 파일명이 환경에 따라 다르게 저장될 수 있어 이름을 한 번 정리해서 비교
# 같은 이름의 기존 파일이 없으면 새로 저장할 경로를 반환
def resolve_nfc_path(save_dir, filename):
    target = unicodedata.normalize('NFC', filename)
    direct = save_dir / filename
    if direct.exists():
        return direct
    for cand in save_dir.iterdir():
        if unicodedata.normalize('NFC', cand.name) == target:
            return cand
    return save_dir / target

In [ ]:
# 한 언론사의 기간 통합 링크 JSON을 읽어 본문을 수집하고 통합 CSV로 저장
# 중간 저장 파일로 이어서 수집하고, 오류가 난 기사는 한 번 더 시도한 뒤 실패 목록 저장
def collect_bodies_selenium(press, start_date, end_date, driver=None, save_dir=SAVE_DIR):
    # 크롬 창을 따로 넘기지 않으면 위 셀에서 만들어 둔 driver 사용
    # 창이 꺼져서 build_driver()로 다시 만든 경우에도 새 창을 잡음
    if driver is None:
        driver = globals()['driver']

    # 파일명 키로 쓸 기간 접미사 (예: 260501_260507)
    period = make_period_suffix(start_date, end_date)
    # 입력/출력 경로 — 한글 파일명 차이 때문에 못 찾는 일을 줄이기 위해 이름을 정리해서 비교
    links_path = resolve_nfc_path(save_dir, f"링크_{press}_{period}.json")
    checkpoint_path = resolve_nfc_path(save_dir, f"체크포인트_{BODY_PREFIX}_{press}_{period}.json")
    csv_save_path = resolve_nfc_path(save_dir, f"{BODY_PREFIX}_{press}_{period}.csv")

    # 최종 파일이 이미 있으면 같은 기간은 다시 수집하지 않음
    if SKIP_COMPLETED and csv_save_path.exists():
        print()
        print(f"=== {press} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {csv_save_path}")
        return csv_save_path

    # 링크 파일 불러오기
    if links_path.exists():
        with links_path.open('r', encoding='utf-8') as f:
            naver_news_links = json.load(f)
        # 대략적인 예상 시간 출력 — 실제 시간은 인터넷 상태와 재시도 여부에 따라 달라질 수 있음
        avg_pause_sec = sum(ARTICLE_PAUSE_RANGE_SEC) / 2
        # 크롬으로 페이지를 여는 데 걸리는 시간까지 얹어서 어림잡음
        est_sec_per_article = avg_pause_sec + PAGE_LOAD_WAIT_SEC + 1.2
        est_min = len(naver_news_links) * est_sec_per_article / 60
        print()
        print(f"=== {press} / {start_date} ~ {end_date} Selenium 본문 수집 시작 ===")
        print(f'링크 {len(naver_news_links)}개 불러옴: {links_path}')
        print(f'예상 소요 시간: 약 {est_min:.0f}분')
    else:
        raise FileNotFoundError(f'링크 파일 없음 — 언론사_네이버뉴스_url_수집_selenium_local.ipynb를 먼저 실행하세요\n경로: {links_path}')

    # 이전에 중단된 작업이 있으면 이어받기 — next_i 인덱스 다음부터 시작
    if checkpoint_path.exists():
        with checkpoint_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        # JSON은 dict 키를 문자열로 저장하므로 int로 다시 변환
        all_results = {int(k): v for k, v in checkpoint.get('all_results', {}).items()}
        err_idx = checkpoint.get('err_idx', [])
        i = checkpoint.get('next_i', 0)
        print(f'체크포인트 발견 — {i}번째부터 이어서 시작 (이미 수집: {len(all_results)}건)')
    else:
        all_results = dict()
        i = 0
        err_idx = []
        print('새로 시작')

    # 추출한 링크에 직접 방문하여 크롤링 진행
    for link in naver_news_links[i:]:
        try:
            all_results[i] = extract_article_selenium(driver, link)

            # 진행 중인 건수 표시 코드
            print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% \t error: {len(err_idx)}')

            i += 1

            # 중간저장 — N건마다 저장해서 중간에 끊겨도 이어서 할 수 있게 함
            if i % CHECKPOINT_INTERVAL == 0:
                with checkpoint_path.open('w', encoding='utf-8') as f:
                    json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False, indent=2)
                print(f'체크포인트 저장 — {i}건 완료')

            # 봇 탐지 방지를 위해 다음 기사 요청 전 랜덤한 시간을 기다림
            polite_sleep('다음 기사 전', ARTICLE_PAUSE_RANGE_SEC)

        except Exception as exc:
            print(f'오류 발생 — index {i}: {exc!r}')
            # 실패한 index는 err_idx에 저장한 뒤 마지막에 한 번 더 재시도
            err_idx.append(i)
            i += 1
            # 오류가 난 번호도 빠지지 않게 바로 중간 저장
            with checkpoint_path.open('w', encoding='utf-8') as f:
                json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False, indent=2)

    # 1차 오류 자동 재시도 — 일시적 네트워크 문제나 응답 지연이었던 경우를 한 번 더 시도
    if err_idx:
        print()
        print(f'오류 {len(err_idx)}건 재시도 시작...')
        re_err_idx = []

        for retry_i in err_idx:
            try:
                link = naver_news_links[retry_i]
                all_results[retry_i] = extract_article_selenium(driver, link)
                print(f'재시도 성공 — index {retry_i}')
                polite_sleep('다음 재시도 전', ARTICLE_PAUSE_RANGE_SEC)
            except Exception as exc:
                print(f'재시도 실패 — index {retry_i}: {exc!r}')
                re_err_idx.append(retry_i)

        # 재시도 후에도 실패한 인덱스만 남김 — 재실패 JSON으로 저장 대상
        err_idx = re_err_idx
        print(f'재시도 완료 — 재실패: {len(err_idx)}건')

    # 수집한 정보들을 표 형태로 변환
    df = pd.DataFrame(all_results).T

    if df.empty:
        raise ValueError('수집된 본문 데이터가 없습니다.')

    # 수집한 기사들 중 중복인 경우 이를 제거
    df_no_duplicates = df.drop_duplicates().reset_index(drop=True)

    # 오래된 순부터 수집했으나 혹시 모를 상황을 방지하기 위해 pubdate를 datetime으로 변환 후 정렬
    df_no_duplicates['pubdate'] = pd.to_datetime(df_no_duplicates['pubdate'], errors='coerce')
    df_sorted = df_no_duplicates.sort_values(by='pubdate')

    # 수집한 정보들을 csv로 저장
    df_sorted.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {csv_save_path}')

    if err_idx:
        # 다시 시도해도 실패한 기사 주소는 별도 JSON으로 저장해 나중에 사람이 확인할 수 있게 함
        failed_path = save_dir / f"{BODY_PREFIX}_재실패_{press}_{period}.json"
        with failed_path.open('w', encoding='utf-8') as f:
            json.dump({'err_idx': err_idx, 'links': [naver_news_links[x] for x in err_idx]}, f, ensure_ascii=False, indent=2)
        print(f'재실패 목록 저장: {failed_path}')
    elif checkpoint_path.exists():
        # 재실패가 없을 때만 중간 파일 삭제 — 실패가 있으면 참고용으로 남김
        checkpoint_path.unlink()

    print(f'본문 수집 완료 — 총 {len(df_sorted)}건 / 오류 {len(err_idx)}건')
    return csv_save_path


# 생성된 jobs를 순서대로 실행
# 한 작업이 실패해도 실패 목록에 기록하고 다음 작업으로 넘어감
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        # 작업 정보의 언론사, 시작일, 종료일을 본문 수집 함수에 전달
        results.append(collect_bodies_selenium(driver=driver, **job))
    except Exception as exc:
        # 한 언론사에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어감: {exc!r}")
    finally:
        if index < len(jobs):
            # 다음 언론사로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 다시 돌릴 수 있게 파일로 저장
if failures:
    failures_path = SAVE_DIR / f'{BODY_PREFIX}_수집실패목록_naver.json'
    with failures_path.open('w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)

In [ ]:
# 작업이 끝나면 크롬 창 종료
# 통합 셀은 크롬 없이도 돌아가서 여기서 닫아도 됨
try:
    driver.quit()
    print('브라우저 종료 완료')
except Exception as exc:
    print(f'브라우저 종료 중 오류: {exc!r}')

## press별 본문 CSV 통합

위에서 만든 `본문_selenium_{press}_{기간}.csv` 파일들을 하나로 합치는 단계임.
각 행에 언론사 이름과 매체 그룹을 붙이고, 전처리·분석 단계에서 사용할 통합 CSV 저장.

- 입력: 언론사별 `본문_selenium_{press}_{기간}.csv` (Colab에서 받아 둔 `본문_bs4_*`도 같이 읽음)
- 출력: `통합_본문_bs4_언론사_{기간}.csv` — 뒤 단계(`pipeline_py`)가 찾는 이름이라 그대로 유지
- `_direct`가 붙은 방송사 직접 수집 파일은 통합에서 제외
- 같은 기사 주소가 중복으로 들어온 경우 한 번 더 제거


In [ ]:
# === press별 본문 CSV 통합 (+ media_group 부착) ===
import re
import unicodedata

# press와 media_group을 연결하는 표. 새 매체를 수집하면 여기만 추가
MEDIA_GROUP_MAP = {
    'KBS': '지상파', 'MBC': '지상파', 'SBS': '지상파',
    'YTN': '통신·보도', '연합뉴스': '통신·보도',
    '한국경제': '경제', '매일경제': '경제',
    '조선일보': '정치색', '한겨레': '정치색',
}
EXCLUDE_DIRECT = True  # 방송사 직접 수집(_direct) 결과물은 통합에서 제외


def _nfc(name):
    # 한글 파일명이 환경에 따라 다를 수 있어 비교 전에 이름을 정리
    return unicodedata.normalize('NFC', name)

# SAVE_DIR의 본문_{selenium|bs4}_{press}_{YYMMDD}_{YYMMDD}.csv를 모두 모음
# 로컬 수집물과 Colab에서 받아 둔 파일이 한 폴더에 섞여 있어도 같이 읽힘
# 현재 press_ranges 기간만 통합 (폴더에 다른 기간 파일이 남아 섞이는 것 방지)
target_periods = {make_period_suffix(r['start_date'], r['end_date']) for r in press_ranges}
print(f'통합 대상 기간: {sorted(target_periods)}')

infos = []
for path in SAVE_DIR.iterdir():
    if not path.is_file():
        continue
    m = re.match(r'^본문_(?:selenium|bs4)_(.+)_(\d{6})_(\d{6})\.csv$', _nfc(path.name))
    if not m:
        continue
    press, s, e = m.groups()
    if EXCLUDE_DIRECT and press.endswith('_direct'):
        print(f'제외(_direct): {_nfc(path.name)}')
        continue
    if f'{s}_{e}' not in target_periods:
        print(f'기간 불일치 제외: {_nfc(path.name)} (대상 {sorted(target_periods)})')
        continue
    infos.append((press, s, e, path))

infos.sort(key=lambda x: (x[0], x[1]))
if not infos:
    raise FileNotFoundError(f'통합할 본문_*.csv가 없습니다: {SAVE_DIR}')

frames = []
for press, s, e, path in infos:
    d = pd.read_csv(path, encoding='utf-8-sig')
    d['press'] = press                       # 파일명에서 떼어낸 press를 행마다 부착
    d['source_period'] = f'{s}_{e}'
    d['source_file'] = _nfc(path.name)
    frames.append(d)
    print(f'  {press} {s}_{e}: {len(d)}행')

merged = pd.concat(frames, ignore_index=True)

# media_group 붙이기 — 연결표에 없는 press는 미분류로 두고 경고 (수집 이름과 키를 맞추세요)
merged['media_group'] = merged['press'].map(MEDIA_GROUP_MAP)
unmapped = sorted(merged.loc[merged['media_group'].isna(), 'press'].unique())
if unmapped:
    print(f'\n[경고] media_group 매핑 없는 press: {unmapped} -> 미분류 (MEDIA_GROUP_MAP에 추가하세요)')
merged['media_group'] = merged['media_group'].fillna('미분류')

before = len(merged)
if 'link' in merged.columns:
    merged = merged.drop_duplicates(subset=['link'], keep='first').reset_index(drop=True)
if 'pubdate' in merged.columns:
    merged['pubdate'] = pd.to_datetime(merged['pubdate'], errors='coerce')
    merged = merged.sort_values(['media_group', 'press', 'pubdate']).reset_index(drop=True)

period_suffix = f"{min(s for _, s, _, _ in infos)}_{max(e for _, _, e, _ in infos)}"
combined_path = SAVE_DIR / f'통합_본문_bs4_언론사_{period_suffix}.csv'
merged.to_csv(combined_path, index=False, encoding='utf-8-sig')

print(f'\n통합 저장 완료: {combined_path}')
print(f'행 수: {before} -> {len(merged)} (중복 {before - len(merged)}건 제거)')
print('media_group별:', merged['media_group'].value_counts().to_dict())
print('press별:', merged['press'].value_counts().to_dict())

# 분석 단계(pipeline_py)는 data/news 바로 아래의 통합본을 읽음
# 이번에 만든 통합본으로 분석을 돌리려면 아래 경로로 직접 복사해 쓰면 됨 (기존 파일 덮어쓰기 주의)
print(f"\n분석에 쓰려면 복사: cp '{combined_path}' '{PROJECT_DIR / 'data' / 'news'}/'")